In [0]:
mydata=[(1, 'aa',10),(2,'bb',20),(3,'cc',30)]

myschema="id int,name string,age int"
df=spark.createDataFrame(mydata,myschema)
display(df)
df.show()

id,name,age
1,aa,10
2,bb,20
3,cc,30


+---+----+---+
| id|name|age|
+---+----+---+
|  1|  aa| 10|
|  2|  bb| 20|
|  3|  cc| 30|
+---+----+---+



DB Utilities

In [0]:
path = "abfss://source@datalakegyan.dfs.core.windows.net/"

display(dbutils.fs.ls(path))

path,name,size,modificationTime
abfss://source@datalakegyan.dfs.core.windows.net/Sales.csv,Sales.csv,869537,1791720737000


In [0]:
folder_path = "abfss://source@datalakegyan.dfs.core.windows.net/day71"

dbutils.fs.mkdirs(folder_path)

True

Read a Source File


In [0]:
path = "abfss://source@datalakegyan.dfs.core.windows.net"

df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

df.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

## Databricks dbutils, Widgets & Secrets

In [0]:
source_path = (
    "abfss://source@datalakegyan.dfs.core.windows.net/Sales.csv"
)

destination_path = (
    "abfss://destination@datalakegyan.dfs.core.windows.net/Sales.csv"
)

dbutils.fs.cp(source_path, destination_path)

True

Widgets — Accept Notebook Inputs

In [0]:
dbutils.widgets.text("item_type", "Dairy", "Item Type")

item_type = dbutils.widgets.get("item_type")

input_path = "abfss://source@datalakegyan.dfs.core.windows.net/Sales.csv"

df = spark.read.option(
    "header", True
).option(
    "inferSchema", True
).csv(input_path)

result_df = df.filter(
    df.Item_Type == item_type
)

display(result_df)

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
FDA03,18.5,Regular,0.045463773,Dairy,144.1102,OUT046,1997,Small,Tier 1,Supermarket Type1,2187.153
FDU02,13.35,Low Fat,0.10249212,Dairy,230.5352,OUT035,2004,Small,Tier 2,Supermarket Type1,2748.4224
FDE51,5.925,Regular,0.161466534,Dairy,45.5086,OUT010,1998,null,Tier 3,Grocery Store,178.4344
FDV38,19.25,Low Fat,0.170348551,Dairy,55.7956,OUT010,1998,null,Tier 3,Grocery Store,163.7868
FDU02,13.35,Low Fat,0.102511504,Dairy,230.6352,OUT046,1997,Small,Tier 1,Supermarket Type1,3435.528
FDM39,6.42,LF,0.089498926,Dairy,178.1002,OUT010,1998,null,Tier 3,Grocery Store,358.2004
FDS02,null,Regular,0.255394896,Dairy,196.8794,OUT019,1985,Small,Tier 1,Grocery Store,780.3176
FDZ03,13.65,Regular,0.078946455,Dairy,186.024,OUT045,2002,null,Tier 2,Supermarket Type1,1118.544
DRG27,8.895,Low Fat,0.105274111,Dairy,39.9138,OUT049,1999,Medium,Tier 1,Supermarket Type1,690.4346


PySpark Data Ingestion in Azure Databricks

In [0]:
source_path = "abfss://source@datalakegyan.dfs.core.windows.net/Sales.csv"

sales_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(source_path)

#sales_df.printSchema()
total_rows = sales_df.count()

print("Total rows:", total_rows)

Total rows: 8523


##Delta Table


In [0]:
dbutils.fs.rm("abfss://destination@datalakegyan.dfs.core.windows.net/Sales.csv")
dbutils.fs.ls("abfss://destination@datalakegyan.dfs.core.windows.net/")


[]

In [0]:
source_path="abfss://source@datalakegyan.dfs.core.windows.net/Sales.csv"

sales_df=spark.read.option("Header",True).option("inferSchema",True).csv(source_path)

destination_path="abfss://destination@datalakegyan.dfs.core.windows.net/Sales.csv"

delta_path=sales_df.write.format("delta").mode("overwrite").save(destination_path)

delta_df = spark.read.format("delta").load(destination_path)

display(delta_df.limit(5))

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [0]:
%sql
SELECT * from databricksmaster.information_schema.columns;

table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,full_data_type,data_type,character_maximum_length,character_octet_length,numeric_precision,numeric_precision_radix,numeric_scale,datetime_precision,interval_type,interval_precision,maximum_cardinality,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_system_time_period_start,is_system_time_period_end,system_time_period_timestamp_generation,is_updatable,partition_index,comment
databricksmaster,information_schema,abac_policy_definitions,policy_id,0,null,NO,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,Unique identifier of the policy.
databricksmaster,information_schema,abac_policy_definitions,policy_name,1,null,NO,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,Name of the policy.
databricksmaster,information_schema,abac_policy_definitions,policy_type,2,null,NO,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,"The policy type. For example, ROW_FILTER, COLUMN_MASK, GRANT, or DENY."
databricksmaster,information_schema,abac_policy_definitions,catalog_name,3,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,Name of the catalog the policy is attached to (or that contains the schema/table the policy is attached to).
databricksmaster,information_schema,abac_policy_definitions,schema_name,4,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,Name of the schema the policy is attached to (or that contains the table the policy is attached to). NULL for CATALOG-attached policies.
databricksmaster,information_schema,abac_policy_definitions,securable_name,5,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,Name of the securable the policy is attached to. NULL for CATALOG-attached and SCHEMA-attached policies.
databricksmaster,information_schema,abac_policy_definitions,on_securable_type,6,null,NO,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,"Type of the securable the policy is attached to. For example, CATALOG, SCHEMA, or TABLE."
databricksmaster,information_schema,abac_policy_definitions,to_principals,7,null,NO,array,ARRAY,null,null,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,"List of principals (users, groups, service principals) the policy applies to."
databricksmaster,information_schema,abac_policy_definitions,except_principals,8,null,NO,array,ARRAY,null,null,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,List of principals excluded from the policy.
databricksmaster,information_schema,abac_policy_definitions,for_securable_type,9,null,NO,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,Type of securable the policy targets when evaluated.


CRUD Operations


In [0]:
%sql
create table sales 
using DELTA
LOCATION "abfss://destination@datalakegyan.dfs.core.windows.net/Sales.csv"

In [0]:
%sql
select * from sales limit 5;

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [0]:
%sql DESCRIBE HISTORY sales

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-18T09:06:44.000Z,146890828234657,suryavanshigyaneshwar@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(702785274900741),133407da-d915-4e12-b54c-935d5038efba,1018-080015-8mpxbxll-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 206280, numDeletionVectorsRemoved -> 0, numOutputRows -> 8523, numOutputBytes -> 206280)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-18T09:06:29.000Z,146890828234657,suryavanshigyaneshwar@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(702785274900741),9a239b9d-0128-4247-af8e-db8ffe705acd,1018-080015-8mpxbxll-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 206280, numDeletionVectorsRemoved -> 0, numOutputRows -> 8523, numOutputBytes -> 206280)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-18T09:04:16.000Z,146890828234657,suryavanshigyaneshwar@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(702785274900741),ddff3b1d-5dda-44ca-b9f1-0acb435f7686,1018-080015-8mpxbxll-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 8523, numOutputBytes -> 206280)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT *
FROM sales VERSION AS OF 0
LIMIT 5;

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


VACCUM+ZORDER+OPTIMIZE

In [0]:
%sql OPTIMIZE sales



path,metrics
abfss://destination@datalakegyan.dfs.core.windows.net/Sales.csv,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1792598209338, 1792598211439, 8, 0, null, List(0, 0), null, 12, 12, 0, 0, null, null, 0)"


In [0]:
%sql

OPTIMIZE sales
ZORDER BY (Item_MRP);



path,metrics
abfss://destination@datalakegyan.dfs.core.windows.net/Sales.csv,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 206280), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1792598252823, 1792598253909, 8, 0, null, List(0, 0), null, 12, 12, 0, 0, null, null, 0)"


In [0]:
%sql
VACUUM sales DRY RUN;

path


AutoLoader /Streaming DataFrame

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format', 'parquet')\
    .option('cloudFiles.schemaLocation', 'abfss://aldestination@datalakegyan.dfs.core.windows.net/checkpoint')\
    .load('abfss://alsource@datalakegyan.dfs.core.windows.net')

df.writeStream\
    .format('delta')\
    .option('checkpointLocation', 'abfss://aldestination@datalakegyan.dfs.core.windows.net/checkpoint')\
    .option('mergeSchema', 'true')\
    .trigger(availableNow=True)\
    .start('abfss://aldestination@datalakegyan.dfs.core.windows.net/data')
